---
title: "Лабораторна робота №1. Ухвалення рішень в умовах невизначеності. Частина 1"
description: "Моніторинг та керування в слабкоструктурованих процесах і системах | КрНУ ім. М. Остроградського"
author: "Ярослав Фігас, 2026"
date: today
lang: uk
jupyter: python3

format:
  html:
    toc: true
    toc-location: right
    number-sections: false
    code-fold: true
    embed-resources: true
    self-contained-math: true
---

# Мета роботи
Опанувати базову матричну постановку задачі вибору та навчитися застосовувати класичні критерії прийняття рішень в умовах невизначеності: *Вальда*, *оптимізму (maximax)*, *Байєса–Лапласа* і *Севіджа*.

# Індивідуальне завдання

## Умови завдання

Було наведено заготовку для *власної матриці виграшів*.
Є можливість:

- залишити таку саму розмірність, як у базовому прикладі;
- змінити назви альтернатив і станів;
- використати власні значення виграшів;
- за потреби задати свій розподіл імовірностей.

Рекомендується пов'язати матрицю з реальною задачею моніторингу або керування, наприклад:

- вибір стратегії реагування на інцидент;
- вибір режиму моніторингу;
- вибір конфігурації аналітичної системи;
- вибір дії адміністратора в умовах нестабільного інформаційного фону.

## Виконання роботи

Для початку потрібно імпортувати потрібні бібліотеки, які необхідні для роботи. Numpy - для обчислень, а pandas - для роботи з данними.

In [26]:
import numpy as np
import pandas as pd

Наступним кроком буде створення матриці виграшів. Для роботи була надана заготвля, яку можна підкорегувати під свій лад. Тематика була обрана на тему "Стратегія реагування на інцедент". Було додано 4 альтернативи та стани, а також ймовірності станів.

In [27]:

student_alternatives = [
    "Моніторинг ситуації",
    "Обмеження доступу",
    "Резервна інфраструктура",
    "Блокування інфраструктури"
]

student_states = [
    "Відсутність загрози",
    "Низька загроза",
    "Середня загроза",
    "Велика загроза"
]

student_matrix = np.array([
    [10,  4, -2, -10],
    [ 6,  7,  1, -5],
    [ 1,  3,  7,  2],
    [ -7, -4 , 3, 10]
], dtype=float)

student_probabilities = np.array([0.40, 0.30, 0.25, 0.05], dtype=float)

student_df = pd.DataFrame(student_matrix, index=student_alternatives, columns=student_states)
student_df

,Відсутність загрози,Низька загроза,Середня загроза,Велика загроза
Моніторинг ситуації,10.0,4.0,-2.0,-10.0
Обмеження доступу,6.0,7.0,1.0,-5.0
Резервна інфраструктура,1.0,3.0,7.0,2.0
Блокування інфраструктури,-7.0,-4.0,3.0,10.0


Наступним кроком буде робота з різними критеріями. Перегляд критеріїв почнімо з критерію Вальда, який шукає для кожної альтернативи найгірший варіант, і серед них обирається найкращий.

In [28]:
def wald_criterion(matrix, alternatives):
    rows = matrix.min(axis=1)
    best = rows.argmax()
    return pd.DataFrame({
        "Альтернатива": alternatives,
        "Найгірший варіант": rows
    }), alternatives[best], rows[best]
wald_table, wald_best_alt, wald_best_value = wald_criterion(student_matrix, student_alternatives)

display(wald_table)
print(f"Оптимальна альтернатива за критерієм Вальда: {wald_best_alt}")
print(f"Оцінка критерію: {wald_best_value}")

,Альтернатива,Найгірший варіант
0,Моніторинг ситуації,-10.0
1,Обмеження доступу,-5.0
2,Резервна інфраструктура,1.0
3,Блокування інфраструктури,-7.0


Оптимальна альтернатива за критерієм Вальда: Резервна інфраструктура
Оцінка критерію: 1.0


З результатів можна помітити, що серед усіх альтернатив, найкращим виявилась резервна інфраструктура, оскільки серед усіх найгірших результатів, даний вибір завжди вийде в плюс, які б обставини не були.
Наступним кроком розглянемо критерій оптимізму. Цей критерій, навпаки, шукає найвищу вигоду.

In [29]:
def maximax_criterion(matrix, alternatives):
    rows = matrix.max(axis=1)
    best = rows.argmax()
    return pd.DataFrame({
        "Альтернатива": alternatives,
        "Найкращий варіант": rows
    }), alternatives[best], rows[best]
maximax_table, maximax_best_alt, maximax_best_value = maximax_criterion(student_matrix, student_alternatives)

display(maximax_table)
print(f"Оптимальна альтернатива за критерієм оптимізму: {maximax_best_alt}")
print(f"Оцінка критерію: {maximax_best_value}")

,Альтернатива,Найкращий варіант
0,Моніторинг ситуації,10.0
1,Обмеження доступу,7.0
2,Резервна інфраструктура,7.0
3,Блокування інфраструктури,10.0


Оптимальна альтернатива за критерієм оптимізму: Моніторинг ситуації
Оцінка критерію: 10.0


З даного результату у нас є два найкращих варіанти: моніторинг та блокування. Проте найкращий результат не є накращим варіантом, оскільки при високій загрозі інфраструктурі, моніторінг не сильно принесе пользи, як і від блокуквання інфраструктури при відсутності загрози, яке понесе значний збиток.
Наступним буде критерій Баєеса-Лапласа, який використовує ймовірність події стану і обчислює очікуваний виграш.

In [30]:
def bayes_laplace_criterion(matrix, alternatives, probs):
    expected = matrix @ probs
    best = expected.argmax()
    return pd.DataFrame({
        "Альтернатива": alternatives,
        "Математичне сподівання": expected
    }), alternatives[best], expected[best]
bayes_table, bayes_best_alt, bayes_best_value = bayes_laplace_criterion(
    student_matrix,
    student_alternatives,
    student_probabilities
)

display(bayes_table)
print(f"Оптимальна альтернатива за критерієм Байєса–Лапласа: {bayes_best_alt}")
print(f"Оцінка критерію: {bayes_best_value:.3f}")

,Альтернатива,Математичне сподівання
0,Моніторинг ситуації,4.20
1,Обмеження доступу,4.50
2,Резервна інфраструктура,3.15
3,Блокування інфраструктури,-2.75


Оптимальна альтернатива за критерієм Байєса–Лапласа: Обмеження доступу
Оцінка критерію: 4.500


Враховуючи ймовірність на настання певного стану, то даний метод обрав найоптимальніший варіант з обмеженням доступу, з найбільшим математичним сподіванням.
Останній критерій який буде розглянуто, це Критерій Севіджа який працює з жалем, тобто з втратами від того, що ми не обрали найкращу альтернативу для конкретного стану середовища.

In [31]:
def savage_criterion(matrix, alternatives, states):
    column_maxs = matrix.max(axis=0)
    regret_matrix = column_maxs - matrix
    row_regret_maxs = regret_matrix.max(axis=1)
    best_index = row_regret_maxs.argmin()

    regret_df = pd.DataFrame(regret_matrix, index=alternatives, columns=states)
    summary_df = pd.DataFrame({
        "Альтернатива": alternatives,
        "Маскисмальний жаль": row_regret_maxs
    })
    return regret_df, summary_df, alternatives[best_index], row_regret_maxs[best_index]
regret_df, savage_table, savage_best_alt, savage_best_value = savage_criterion(
    student_matrix,
    student_alternatives,
    student_states
)

print("Матриця жалю:")
display(regret_df)

print("Підсумкова таблиця:")
display(savage_table)

print(f"Оптимальна альтернатива за критерієм Севіджа: {savage_best_alt}")
print(f"Оцінка критерію: {savage_best_value}")

Матриця жалю:


,Відсутність загрози,Низька загроза,Середня загроза,Велика загроза
Моніторинг ситуації,0.0,3.0,9.0,20.0
Обмеження доступу,4.0,0.0,6.0,15.0
Резервна інфраструктура,9.0,4.0,0.0,8.0
Блокування інфраструктури,17.0,11.0,4.0,0.0


Підсумкова таблиця:


,Альтернатива,Маскисмальний жаль
0,Моніторинг ситуації,20.0
1,Обмеження доступу,15.0
2,Резервна інфраструктура,9.0
3,Блокування інфраструктури,17.0


Оптимальна альтернатива за критерієм Севіджа: Резервна інфраструктура
Оцінка критерію: 9.0


Для зручності зведемо усі результати в одну таблицю. У таблиці показуються різні оптимальні альтернативи та різні оцінки.

In [32]:
summary_df = pd.DataFrame({
    "Критерій": ["Вальда", "Оптимізму (maximax)", "Байєса–Лапласа", "Севіджа"],
    "Оптимальна альтернатива": [wald_best_alt, maximax_best_alt, bayes_best_alt, savage_best_alt],
    "Оцінка": [wald_best_value, maximax_best_value, round(bayes_best_value, 3), savage_best_value]
})

summary_df

,Критерій,Оптимальна альтернатива,Оцінка
0,Вальда,Резервна інфраструктура,1.0
1,Оптимізму (maximax),Моніторинг ситуації,10.0
2,Байєса–Лапласа,Обмеження доступу,4.5
3,Севіджа,Резервна інфраструктура,9.0


# Висновки

У цій лабораторній роботі було проведено аналіз матриці виграшів на тему "Стратегія реагування на інцедент" розміром 4 на 4. Для кожного зі станів у даній матриці було додано свою ймовірність, де на кращу подію ймовірність найвища і спадала сильніще в залежності від критичності ситуації.
Під час роботи було досліджено 4 критерії. В критерії Вальда найкращим варіантом виявилась Критична інфраструктура, оскільки з найгірших варіантів цей виявився найкращим. В критерії оптимізму найкращими варіантами стали брокування інфраструктури та спостереження, оскільки вони несуть найбільший виграш. В критерії Байєса-Лапласа найкращим варіантом став обмеження доступу, оскільки враховуючи ймовірності відбування події, цей варіант вийшов найкращим. І також за критерієм Севіджа найкращим варіантом виявився резервна інфраструктура, оскільки показник жалю виявився найнижчим.
Що до використання критеріїв то критерій Вальда корисний у прорахунках найгірших варіантів для уникнення значних втрат, критерій оптимізму підходить для ризикованих дій із потенційно найбільшим виграшем і для короткострокового виграшу це добрий варіант аналітики, критерій Байєса-Лапласа працює навпаки, тобто розрахований на довгостроковий виграш, оскільки він полагається на математичне сподівання що є реалістичним, а критерій Севіджа дозволяє оцінити розмір втрат при виборі певного варіанту і корисний для додаткової оцінки втрат.
На мою думку найстійкішою альтернативною вважається омбеження доступу, оскільки для вирахування цього варіанту використовувася критерій Байєса-Лапласа, який включає в аналіз ймовірність того, що подія відбудеться. 

# Контрольні питання

Що означає подати задачу вибору у вигляді *матриці виграшів*?

Подати задачу вибору у вигляді матриці виграшів — це означає структурувати її у вигляді таблиці, яка показує, які результати отримає особа, що приймає рішення (ОПР), залежно від її власного вибору та зовнішніх обставин.

Яку позицію щодо невизначеності реалізує критерій *Вальда*?

Критерій Вальда використовує стратегію низьких ризиків та ворожого середовища, тобто потрібно отримати вигоду бажано з відсутніми втратами та низькими ризиками. 

Чим критерій *оптимізму (maximax)* відрізняється від критерію *Вальда*?

Критерій оптимізму шукає найкращий варіант чи результат, в той час як критерій Вальда шукає найкращий вихід з найгіршої ситуації.

У чому полягає <u>зміст</u> критерію *Байєса–Лаплас*?

Зміст критерію Байєса-Лапласа полягає у тому, що найкращий варіант обирається з використанням математичного сподівання, і враховує як успіх так і можливий провал.

Що таке *матриця жалю* в критерії <u>Севіджа</u>?

Матриця жалю в критерії Севіджа це таблиця, яка показує втрати при виборі певного рішення. 

Чому різні критерії можуть вказувати на різні альтернативи?

Тому що різні критерії враховують різні обставини і розраховують результати під різні потреби наприклад пошук найкращого результату, або пошук найменшої втрати.

У яких прикладних задачах моніторингу такі критерії є корисними?

Різні критерії підходять під різні сфери де потрібен певний результат. Наприклад в бізнесі можна використати критерій максимальної вигоди або мінімальних втрат тобто що більш важливим. 